# Видео → события → LIF

Учебный кадровый эмулятор DVS: видео преобразуется в события, затем они проходят через LIF-слой. Обучения классификатора здесь нет.

Выполняйте ячейки сверху вниз. Первая ячейка автоматически скачает код и пример видео из `Nek1tos01/dvs-emulator`. Адрес уже указан, загружать ZIP не требуется. В обычном Jupyter запускайте ноутбук из папки проекта.


In [ ]:
from pathlib import Path
import io
import os
import subprocess
import sys
import tempfile
import zipfile

REPO_URL = "https://github.com/Nek1tos01/dvs-emulator"

def is_project(path):
    return (path / "read_mp4.py").is_file() and (path / "lif.py").is_file()

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_DIR = next((p for p in candidates if is_project(p)), None)
if PROJECT_DIR is None:
    destination = Path(tempfile.mkdtemp(prefix="dvs_colab_"))
    if REPO_URL.strip():
        if not REPO_URL.startswith("https://github.com/"):
            raise ValueError("REPO_URL должен быть HTTPS-адресом GitHub-репозитория")
        PROJECT_DIR = destination / "repo"
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
    else:
        try:
            from google.colab import files
        except ImportError as error:
            raise RuntimeError("Откройте ноутбук из папки проекта или заполните REPO_URL") from error
        print("Загрузите dvs-emulator-colab.zip, созданный tools/build_colab_zip.py")
        uploaded = files.upload()
        archives = [name for name in uploaded if name.lower().endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Выберите один ZIP-архив проекта")
        with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
            for member in archive.infolist():
                target = (destination / member.filename).resolve()
                if not target.is_relative_to(destination.resolve()):
                    raise ValueError("Недопустимый путь в ZIP")
            archive.extractall(destination)
        candidates = [destination] + [p for p in destination.iterdir() if p.is_dir()]
        PROJECT_DIR = next((p for p in candidates if is_project(p)), None)
    if PROJECT_DIR is None or not is_project(PROJECT_DIR):
        raise ValueError("В выбранном источнике не найдены модули проекта")

PROJECT_DIR = PROJECT_DIR.resolve()
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("Папка проекта:", PROJECT_DIR)


## Зависимости
Установка относится к текущему ядру. PyTorch может работать на CPU; CUDA используется только при её доступности. Эмуляция OpenCV/NumPy остаётся на CPU.

In [ ]:
if os.environ.get("DVS_SKIP_INSTALL") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)

import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import display

from read_mp4 import convert_video
from snn_input import load_metadata, load_event_part, select_frame_events
from visualization import events_to_image
from run_snn import run_lif

get_ipython().run_line_magic("matplotlib", "inline")
print("Python:", sys.version.split()[0], "NumPy:", np.__version__, "PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())


## Конвертация встроенного примера
Для первого запуска используем небольшой участок и уменьшенное разрешение. Изменение разрешения влияет на события. Все временные метки остаются временем исходного видео, а не скоростью исполнения ячеек.

In [ ]:
MAX_FRAMES = 30  # @param {type:"integer"}
MAX_SIDE = 320  # @param {type:"integer"}
CONTRAST_THRESHOLD = 0.2  # @param {type:"number"}
DEVICE = "auto"  # @param ["auto", "cpu", "cuda"]

VIDEO_PATH = PROJECT_DIR / "Videos" / "IMG_7181.MP4"
run_dir = convert_video(VIDEO_PATH, threshold=CONTRAST_THRESHOLD,
                        max_frames=MAX_FRAMES, max_side=MAX_SIDE, preview=False)
info = load_metadata(run_dir)
display(info)


## Исходный кадр рядом с событиями
Слева кадр 1, справа его события: зелёные ON, красные OFF. Цветная картинка не показывает кратность событий — для LIF используются числовые счётчики.

In [ ]:
frame_index = min(1, info["processed_frames"] - 1)
part = load_event_part(run_dir, frame_index // info["chunk_frames"])
selected = select_frame_events(part, frame_index, info["fps"])
event_image = events_to_image(selected, info["height"], info["width"])

cap = cv2.VideoCapture(str(VIDEO_PATH))
try:
    for _ in range(frame_index + 1):
        ok, frame = cap.read()
        if not ok:
            raise ValueError("Не удалось прочитать демонстрационный кадр")
finally:
    cap.release()
frame = cv2.resize(frame, (info["width"], info["height"]), interpolation=cv2.INTER_AREA)
combined = np.hstack((frame, event_image))
plt.figure(figsize=(8, 7))
plt.imshow(cv2.cvtColor(combined, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Исходный кадр | События ON / OFF")
plt.show()


## Последовательность LIF
Два входных канала: число ON и число OFF. Один шаг соответствует `1 / fps`. Пустые кадры тоже проходят через слой; мембрана сохраняется между файлами. Вес события 0.25 выбран для демонстрации. Обучаемых связей в этом примере нет.

In [ ]:
stats = run_lif(run_dir, device=DEVICE)
assert stats["consumed_events"] == info["total_events"]
with open(run_dir / "lif_summary.json", "w", encoding="utf-8") as file:
    json.dump(stats, file, indent=2)
print("Кадров:", stats["processed_frames"])
print("Событий:", stats["consumed_events"], "Спайков:", stats["total_spikes"])
print("Прочитано частей:", stats["part_reads"], "Устройство:", stats["device"])

times = [step["t"] for step in stats["steps"]]
fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
axes[0].plot(times, [step["input_events"] for step in stats["steps"]])
axes[0].set_ylabel("Входные события")
axes[1].plot(times, [step["output_spikes"] for step in stats["steps"]])
axes[1].set_ylabel("Спайки LIF")
axes[1].set_xlabel("Время видео, секунды")
fig.tight_layout()
plt.show()


## Скачать запись
Скачивание включает части событий, manifest и статистику LIF. Файлы среды Colab временные; сохраните архив на компьютер или Google Drive. Повторное выполнение конвертации создаёт новый каталог запуска.

In [ ]:
archive_path = run_dir.parent / (run_dir.name + ".zip")
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(run_dir.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)
print("Архив:", archive_path)
try:
    from google.colab import files
except ImportError:
    print("Jupyter: откройте папку outputs в файловом браузере и скачайте", archive_path.name)
else:
    files.download(str(archive_path))


## Ограничения
Эмулятор не восстанавливает межкадровые события и микросекундные времена. Использует логарифм яркости в оттенках серого видео, единый порог и ограничение нулей единицей. Гамма, автоэкспозиция и компрессия влияют на результат. Модель реального шума и рефрактерности не включена. Число выходных спайков LIF не обязано равняться числу входных событий.